# wrap-forward-fn-generic — worked example 3: wrap_forward_fn passes non-array outputs through

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wrap-forward-fn-generic`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Some forward fns return non-arrays — `argmax().item()` returns an int, `x.shape` returns a `torch.Size`. The wrapper boxes the result only if it is a `torch.Tensor`; otherwise it returns the raw Python value unchanged, so downstream control flow expecting a real int still works.

## Worked solution

We make the wrapper conditional on the output type.

1. Unbox args and call `fwd_fn` as usual.
2. Conditional box: if the result is a `torch.Tensor`, wrap it in `Tensor`; otherwise return it as-is.
3. This lets ops like `lambda x: int(x.argmax().item())` flow a plain int back to the caller instead of erroring inside `Tensor(...)`.

We wrap an argmax-to-int op and a tensor-returning op, printing that the first returns a bare int and the second returns a boxed `Tensor`.

In [ ]:
import torch as t

class Tensor:
    def __init__(self, array):
        self.array = array

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = [a.array if isinstance(a, Tensor) else a for a in args]
        result = fwd_fn(*raw_args, **kwargs)
        if isinstance(result, t.Tensor):
            return Tensor(result)
        return result
    return tensor_func

argmax_int = wrap_forward_fn(lambda x: int(x.argmax().item()))
double = wrap_forward_fn(lambda x: x * 2)
x = Tensor(t.tensor([0.1, 0.9, 0.3]))
print('argmax int:', argmax_int(x), type(argmax_int(x)).__name__)
print('double boxed:', type(double(x)).__name__)